In [1]:
# Notebook bootstrap: download required files from GitHub if missing
from pathlib import Path
from urllib.request import urlretrieve

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/main"

required_files = [
    ("data/metadata/climate_dataset_dublin_core.xml", "data/metadata/climate_dataset_dublin_core.xml"),
    ("data/metadata/climate_dataset_datacite.xml", "data/metadata/climate_dataset_datacite.xml"),
    ("data/metadata/climate_dataset_schemaorg.jsonld", "data/metadata/climate_dataset_schemaorg.jsonld"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")
    else:
        print(f"Already available: {local_path}")

print("Bootstrap complete.")

Already available: data/metadata/climate_dataset_dublin_core.xml
Already available: data/metadata/climate_dataset_datacite.xml
Already available: data/metadata/climate_dataset_schemaorg.jsonld
Bootstrap complete.


# Day 2: Mini Metadata Validation (Teaching Demo)

This notebook presents a **simple, classroom-friendly metadata validation workflow**.

Instead of building a formal validator, we do something more useful for teaching:

1. **Load three metadata records** for the same dataset  
2. **Parse key fields** from each standard  
3. **Check whether important fields are present**  
4. **Compare completeness across standards**  
5. **Discuss what the results mean for FAIR**

---

## Learning goals

By the end of this notebook, students should be able to:

- explain what a **metadata completeness check** is,
- identify **important metadata fields**,
- understand that different standards organize similar information differently,
- connect metadata quality to **Findability** and **Reusability**.

---

## Important note

This is a **teaching demo**, not a formal metadata validator.  
It checks only a few important fields to make the logic easy to understand in class.


## Step 1 — Define the files we want to validate

We will validate three metadata records that describe the same example dataset:

- **Dublin Core XML**
- **DataCite XML**
- **schema.org JSON-LD**

This is useful because students can see that the **same dataset** can be represented in **different metadata standards**.


In [2]:
from pathlib import Path
import json
import xml.etree.ElementTree as ET
import pandas as pd

dc_path = Path("data/metadata/climate_dataset_dublin_core.xml")
datacite_path = Path("data/metadata/climate_dataset_datacite.xml")
schemaorg_path = Path("data/metadata/climate_dataset_schemaorg.jsonld")

files_df = pd.DataFrame(
    [
        {"standard": "Dublin Core", "path": str(dc_path), "exists": dc_path.exists()},
        {"standard": "DataCite", "path": str(datacite_path), "exists": datacite_path.exists()},
        {"standard": "schema.org", "path": str(schemaorg_path), "exists": schemaorg_path.exists()},
    ]
)

files_df

,standard,path,exists
0,Dublin Core,data/metadata/climate_dataset_dublin_core.xml,True
1,DataCite,data/metadata/climate_dataset_datacite.xml,True
2,schema.org,data/metadata/climate_dataset_schemaorg.jsonld,True


## Step 2 — Preview the raw metadata files

Before validating anything, it is helpful to look briefly at the raw files.

This shows students that:
- metadata is stored in different formats,
- XML and JSON-LD look different,
- but both aim to describe the dataset in a structured way.


In [3]:
# Preview a short snippet from each file
preview_length = 500

for label, path in [
    ("Dublin Core", dc_path),
    ("DataCite", datacite_path),
    ("schema.org", schemaorg_path),
]:
    print(f"\n{'='*80}")
    print(label)
    print(f"{'='*80}")
    text = path.read_text(encoding="utf-8")
    print(text[:preview_length] + ("..." if len(text) > preview_length else ""))


Dublin Core
<?xml version="1.0" encoding="UTF-8"?>
<metadata
  xmlns:dc="http://purl.org/dc/elements/1.1/"
  xmlns:dcterms="http://purl.org/dc/terms/">

  <dc:title>2025 Global Climate Data</dc:title>

  <dc:creator>Global Climate Data Team</dc:creator>

  <dc:publisher>Open Research Repository (Teaching)</dc:publisher>

  <dc:identifier>10.1234/gcd-2025</dc:identifier>

  <dcterms:description>
    A small teaching dataset containing fictional annual climate summaries.
    Variables include average temperat...

DataCite
<?xml version="1.0" encoding="UTF-8"?>
<resource
  xmlns="http://datacite.org/schema/kernel-4"
  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
  xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4/metadata.xsd">

  <identifier identifierType="DOI">10.1234/gcd-2025</identifier>

  <creators>
    <creator>
      <creatorName>Global Climate Data Team</creatorName>
    </creator>
  </creators>

  <titles>
    <title>2025

## Step 3 — Create simple parsers

To compare metadata records, we first convert them into a **common Python structure**.

For teaching, we extract only a small set of important fields:

- `title`
- `creator`
- `identifier`
- `description`
- `publisher`
- `date`

These fields are enough to support a simple completeness discussion.


In [4]:
def _local_name(tag):
    """Remove XML namespace and keep only the local tag name."""
    return tag.split('}', 1)[1] if '}' in tag else tag


def parse_dublin_core(path):
    """Parse a small Dublin Core XML record into a normalized dictionary."""
    root = ET.parse(path).getroot()

    def first(name):
        for el in root.iter():
            if _local_name(el.tag) == name:
                text = (el.text or "").strip()
                if text:
                    return text
        return None

    creators = []
    for el in root.iter():
        if _local_name(el.tag) == "creator":
            text = (el.text or "").strip()
            if text:
                creators.append(text)

    return {
        "standard": "Dublin Core",
        "title": first("title"),
        "creator": creators,
        "identifier": first("identifier"),
        "description": first("description"),
        "publisher": first("publisher"),
        "date": first("date"),
    }


def parse_datacite(path):
    """Parse a small DataCite XML record into a normalized dictionary."""
    root = ET.parse(path).getroot()

    out = {
        "standard": "DataCite",
        "title": None,
        "creator": [],
        "identifier": None,
        "description": None,
        "publisher": None,
        "date": None,
    }

    for el in root.iter():
        tag = _local_name(el.tag)
        text = (el.text or "").strip()

        if tag == "title" and text and out["title"] is None:
            out["title"] = text
        elif tag == "creatorName" and text:
            out["creator"].append(text)
        elif tag == "identifier" and text and out["identifier"] is None:
            out["identifier"] = text
        elif tag == "description" and text and out["description"] is None:
            out["description"] = text
        elif tag == "publisher" and text and out["publisher"] is None:
            out["publisher"] = text
        elif tag in ["publicationYear", "date"] and text and out["date"] is None:
            out["date"] = text

    return out


def parse_schemaorg(path):
    """Parse a small schema.org JSON-LD record into a normalized dictionary."""
    data = json.loads(path.read_text(encoding="utf-8"))

    # Handle both a direct Dataset object and a list/@graph structure
    dataset = data
    if isinstance(data, list):
        dataset = next((item for item in data if item.get("@type") == "Dataset"), data[0])
    elif "@graph" in data:
        dataset = next((item for item in data["@graph"] if item.get("@type") == "Dataset"), data["@graph"][0])

    creators = dataset.get("creator")
    if isinstance(creators, dict):
        creators = [creators.get("name")] if creators.get("name") else []
    elif isinstance(creators, list):
        normalized = []
        for item in creators:
            if isinstance(item, dict) and item.get("name"):
                normalized.append(item.get("name"))
            elif isinstance(item, str):
                normalized.append(item)
        creators = normalized
    elif isinstance(creators, str):
        creators = [creators]
    else:
        creators = []

    publisher = dataset.get("publisher")
    if isinstance(publisher, dict):
        publisher = publisher.get("name")

    description = dataset.get("description")
    identifier = dataset.get("identifier") or dataset.get("@id")
    date = dataset.get("datePublished") or dataset.get("dateModified")

    return {
        "standard": "schema.org",
        "title": dataset.get("name"),
        "creator": creators,
        "identifier": identifier,
        "description": description,
        "publisher": publisher,
        "date": date,
    }

## Step 4 — Parse the three records

Now we run the parsers and inspect the normalized results.

This step is important because it converts **different metadata formats** into a structure that is easy to compare.


In [5]:
parsed_records = [
    parse_dublin_core(dc_path),
    parse_datacite(datacite_path),
    parse_schemaorg(schemaorg_path),
]

parsed_records

[{'standard': 'Dublin Core',
  'title': '2025 Global Climate Data',
  'creator': ['Global Climate Data Team'],
  'identifier': '10.1234/gcd-2025',
  'description': 'A small teaching dataset containing fictional annual climate summaries.\n    Variables include average temperature, rainfall, and CO2 emissions for two fictional countries (Alandia and Borealia) from 2021 to 2023.',
  'publisher': 'Open Research Repository (Teaching)',
  'date': '2025-12-31'},
 {'standard': 'DataCite',
  'title': '2025 Global Climate Data',
  'creator': ['Global Climate Data Team'],
  'identifier': '10.1234/gcd-2025',
  'description': 'A small teaching dataset containing fictional annual climate summaries.\n      Variables include average temperature, rainfall, and CO2 emissions for two fictional countries\n      (Alandia and Borealia) from 2021 to 2023.',
  'publisher': 'Open Research Repository (Teaching)',
  'date': '2025'},
 {'standard': 'schema.org',
  'title': '2025 Global Climate Data',
  'creator': 

In [6]:
parsed_df = pd.DataFrame(parsed_records)
parsed_df

,standard,title,creator,identifier,description,publisher,date
0,Dublin Core,2025 Global Climate Data,[Global Climate Data Team],10.1234/gcd-2025,A small teaching dataset containing fictional ...,Open Research Repository (Teaching),2025-12-31
1,DataCite,2025 Global Climate Data,[Global Climate Data Team],10.1234/gcd-2025,A small teaching dataset containing fictional ...,Open Research Repository (Teaching),2025
2,schema.org,2025 Global Climate Data,[Global Climate Data Team],10.1234/gcd-2025,A small teaching dataset containing fictional ...,Open Research Repository (Teaching),None


## Step 5 — Focus on the most important fields

For class presentation, it is often easier to inspect the key fields one by one.

This table helps students quickly see:
- what is present,
- what is missing,
- and where values are richer or more specific.


In [7]:
display_columns = ["standard", "title", "creator", "identifier", "description", "publisher", "date"]
parsed_df[display_columns]

,standard,title,creator,identifier,description,publisher,date
0,Dublin Core,2025 Global Climate Data,[Global Climate Data Team],10.1234/gcd-2025,A small teaching dataset containing fictional ...,Open Research Repository (Teaching),2025-12-31
1,DataCite,2025 Global Climate Data,[Global Climate Data Team],10.1234/gcd-2025,A small teaching dataset containing fictional ...,Open Research Repository (Teaching),2025
2,schema.org,2025 Global Climate Data,[Global Climate Data Team],10.1234/gcd-2025,A small teaching dataset containing fictional ...,Open Research Repository (Teaching),None


## Step 6 — Define a simple validation checklist

We now create a **lightweight validation rule set**.

We check whether each metadata record contains:

- a **title**
- at least one **creator**
- an **identifier**
- a **description**
- a **publisher**
- a **date**

These are not the only important fields, but they are enough for a useful classroom demonstration.


In [8]:
def is_present(value):
    """Return True if a metadata value is meaningfully present."""
    if value is None:
        return False
    if isinstance(value, str):
        return value.strip() != ""
    if isinstance(value, list):
        return len([v for v in value if v not in [None, ""]]) > 0
    return True


required_fields = ["title", "creator", "identifier", "description", "publisher", "date"]

validation_rows = []
for record in parsed_records:
    row = {"standard": record["standard"]}
    for field in required_fields:
        row[field] = is_present(record.get(field))
    validation_rows.append(row)

validation_df = pd.DataFrame(validation_rows)
validation_df

,standard,title,creator,identifier,description,publisher,date
0,Dublin Core,True,True,True,True,True,True
1,DataCite,True,True,True,True,True,True
2,schema.org,True,True,True,True,True,False


## Step 7 — Calculate a simple completeness score

To make the validation results easier to interpret, we convert the checklist into a simple score.

### Formula
\[
\text{Completeness Score} = \frac{\text{Number of Present Fields}}{\text{Total Checked Fields}} \times 100
\]

Again, this is just a teaching score, not an official standard.


In [9]:
score_rows = []

for record in parsed_records:
    present_count = sum(is_present(record.get(field)) for field in required_fields)
    total_count = len(required_fields)
    score_percent = round((present_count / total_count) * 100, 1)

    score_rows.append({
        "standard": record["standard"],
        "present_fields": present_count,
        "total_checked_fields": total_count,
        "completeness_score_percent": score_percent,
    })

score_df = pd.DataFrame(score_rows).sort_values("completeness_score_percent", ascending=False)
score_df

,standard,present_fields,total_checked_fields,completeness_score_percent
0,Dublin Core,6,6,100.0
1,DataCite,6,6,100.0
2,schema.org,5,6,83.3


## Step 8 — Show missing fields clearly

A completeness score is useful, but students also need to see **which specific fields are missing**.

This makes the notebook more actionable.


In [10]:
missing_rows = []

for record in parsed_records:
    missing = [field for field in required_fields if not is_present(record.get(field))]
    missing_rows.append({
        "standard": record["standard"],
        "missing_fields": ", ".join(missing) if missing else "None",
        "number_of_missing_fields": len(missing),
    })

missing_df = pd.DataFrame(missing_rows).sort_values("number_of_missing_fields")
missing_df

,standard,missing_fields,number_of_missing_fields
0,Dublin Core,None,0
1,DataCite,None,0
2,schema.org,date,1


## Step 9 — Interpret the results from a FAIR perspective

Metadata quality supports FAIR, especially:

### Findable
A dataset is easier to find when it has:
- a clear **title**
- a stable **identifier**
- consistent descriptive fields

### Reusable
A dataset is easier to reuse when it has:
- a meaningful **description**
- clear **creator/publisher** information
- useful **date/context** information

A record may still be valid even if some fields are missing, but weaker metadata usually reduces discoverability and reuse.


In [11]:
fair_interpretation = pd.DataFrame(
    [
        {
            "FAIR principle": "Findable",
            "Most relevant fields in this demo": "title, identifier",
            "Why they matter": "They help users and systems discover and reference the dataset."
        },
        {
            "FAIR principle": "Reusable",
            "Most relevant fields in this demo": "description, creator, publisher, date",
            "Why they matter": "They help others understand the dataset, its context, and how to cite or interpret it."
        },
    ]
)

fair_interpretation

,FAIR principle,Most relevant fields in this demo,Why they matter
0,Findable,"title, identifier",They help users and systems discover and refer...
1,Reusable,"description, creator, publisher, date","They help others understand the dataset, its c..."


## Step 10 — Classroom discussion: strengths, weaknesses, and improvement ideas

This final step is useful for in-class discussion.

Students should move beyond the score and ask:

- Which metadata record looks strongest?
- Which missing fields matter most?
- How would we improve the weakest record?
- Does a high completeness score always mean high metadata quality?

A field can be present but still be weak.  
For example, a very short description may technically exist, but it may still be insufficient for reuse.


In [12]:
improvement_rows = []

for record in parsed_records:
    suggestions = []
    if not is_present(record.get("title")):
        suggestions.append("Add a clear dataset title")
    if not is_present(record.get("creator")):
        suggestions.append("Add creator information")
    if not is_present(record.get("identifier")):
        suggestions.append("Add a persistent identifier")
    if not is_present(record.get("description")):
        suggestions.append("Add a richer description")
    if not is_present(record.get("publisher")):
        suggestions.append("Add publisher/source information")
    if not is_present(record.get("date")):
        suggestions.append("Add publication or creation date")

    improvement_rows.append({
        "standard": record["standard"],
        "improvement_suggestions": "; ".join(suggestions) if suggestions else "Metadata looks complete in this simple checklist"
    })

improvement_df = pd.DataFrame(improvement_rows)
improvement_df

,standard,improvement_suggestions
0,Dublin Core,Metadata looks complete in this simple checklist
1,DataCite,Metadata looks complete in this simple checklist
2,schema.org,Add publication or creation date


## Final takeaway

This notebook demonstrates a simple but effective idea:

> **Metadata validation is not only about checking syntax. It is also about checking whether the metadata is useful enough to support discovery and reuse.**

### In one sentence:
- **More complete metadata** usually supports **better FAIR readiness**, especially for **Findable** and **Reusable**.

### But remember:
- completeness is only one part of quality,
- clarity and usefulness also matter,
- and different metadata standards serve different purposes.
